## LLM Inference & Response Evaluation

We will demonstrate how an LLM is run for inference, prompted, and evaluated using standard response-quality metrics

In [ ]:
# !pip install -U evaluate transformers huggingface_hub rouge_score bert_score

In [1]:
# Setup client
from huggingface_hub import InferenceClient
import os
from dotenv import load_dotenv

load_dotenv(override=True)

HF_TOKEN = os.getenv("HF_TOKEN")

hf_client = InferenceClient(token=HF_TOKEN)

#### Response Generation

In [3]:
HF_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"

def generate(user_input, sys_msg = "You are a helpful professional assistant."):

    # Constructing the message payload
    messages = [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_input}
    ]
    response = hf_client.chat_completion(
                                        model=HF_MODEL,
                                        messages=messages,)

    return response.choices[0].message.content

### Prompt set for evaluation

In [5]:
# Let's load a file for long-text summarisation task
file_path = 'data/sample_doc.txt'

with open(file_path, 'r') as file:
    file_content = file.read()

# print(file_content)

In [6]:
# Some example prompts
prompts = {
    "summarisation": "Summarise the following board memo in 3 bullet points:" + file_content,
    "reasoning": "A company sees falling margins despite rising revenue. List 3 plausible causes.",
    "instruction_following": "Draft a professional email declining a vendor proposal politely.",
    "factual": "What is the difference between revenue and profit?"
}

#### Responses

In [7]:
responses = {}

for task, prompt in prompts.items():
    responses[task] = generate(prompt)

responses

{'summarisation': 'Here are 3 bullet points summarizing the Board Memorandum:\n\n• **Organisational Performance and Market Context**: The organisation delivered solid performance in FY2025, but with uneven results and under pressure from cost inflation, competitive intensity, and customer price sensitivity. Market conditions remain challenging, with demand stabilizing but not returning to pre-volatility growth trajectories.\n\n• **Priorities for FY2026–FY2028**: Management proposes three overarching priorities: 1) operational resilience and efficiency to protect margins and support sustainable growth, 2) focused investment in data, digital, and capability with a clear line of sight to value creation, and 3) simplification of the operating model to enable faster decision-making and improved accountability.\n\n• **Challenges and Trade-offs**: The organisation will face difficult trade-offs in pursuing priorities, including growth opportunities, exiting legacy activities, and de-prioritiz

### Evaluation metrics

LLM evaluation is not just accuracy in the classical ML sense
- Surface-level quality (overlap based)
- Semantic similarity (meaning based)
- Human interpretable checks


Given the same input, multiple responses can be equally valid but differ in wording, structure, or level of detail. Evaluation focuses on analysing model behaviour under a fixed prompt and decoding setup, rather than judging outputs as strictly right or wrong.

The objective is to understand how closely model responses align with reference expectations and where they diverge.

##### Types of Evaluation

LLM evaluation is commonly approached in three ways.

* **Qualitative evaluation** relies on human judgement to assess relevance, coherence, and factual soundness.
* **Reference-based automated evaluation** compares generated outputs against one or more human-written references to estimate similarity.
* **Task or system-level evaluation** examines consistency, robustness, and failure modes across multiple inputs rather than individual responses.

##### Evaluation Metrics

* **BLEU** – Measures n-gram precision between generated text and reference text; sensitive to exact wording and penalises paraphrasing.
* **ROUGE** – Measures n-gram overlap with a focus on recall; commonly used for summarisation tasks.
* **BERTScore** – Measures semantic similarity using contextual embeddings; more robust to paraphrasing and rewording.

These metrics capture different aspects of similarity and are best interpreted together, not individually.

Reference answers for demo evaluation

In [8]:
reference_answers = {
    "reasoning": (
        "Operating or input costs have increased faster than revenue growth, reducing margins. "
        "Revenue growth is driven by discounts, promotions, or a shift toward lower-margin products. "
        "Fixed costs or overheads have expanded without a proportional improvement in efficiency."
    ),
    "instruction_following": (
        "Dear [Vendor Name], Thank you for sharing your proposal and for the time and effort your team "
        "has invested. After careful consideration, we have decided not to proceed at this stage, as the "
        "proposal does not fully align with our current priorities. We appreciate the opportunity to "
        "review your offering and wish you continued success. We will be happy to reconnect if there is "
        "a better fit in the future. Kind regards, [Your Name]"
    ),
    "factual": (
        "Revenue is the total income generated from a company's business activities, while profit is the "
        "amount remaining after all costs and expenses have been deducted."
    )
}

In [9]:
# Prepare predictions and references
tasks = ["reasoning", "instruction_following", "factual"]

predictions = [responses[t] for t in tasks]
references = [reference_answers[t] for t in tasks]

#### Qualitative evaluation rubrics (Manual)

Score each response on:
- **Relevance** (Does it answer the question?)
- **Clarity** (Is it readable and structured?)
- **Risk** (Hallucination, tone, compliance)

We will evaluate the summarisation task manually

In [10]:
print(responses['summarisation'])
# print(responses['instruction_following']) # try printing responses from other tasks as well

Here are 3 bullet points summarizing the Board Memorandum:

• **Organisational Performance and Market Context**: The organisation delivered solid performance in FY2025, but with uneven results and under pressure from cost inflation, competitive intensity, and customer price sensitivity. Market conditions remain challenging, with demand stabilizing but not returning to pre-volatility growth trajectories.

• **Priorities for FY2026–FY2028**: Management proposes three overarching priorities: 1) operational resilience and efficiency to protect margins and support sustainable growth, 2) focused investment in data, digital, and capability with a clear line of sight to value creation, and 3) simplification of the operating model to enable faster decision-making and improved accountability.

• **Challenges and Trade-offs**: The organisation will face difficult trade-offs in pursuing priorities, including growth opportunities, exiting legacy activities, and de-prioritizing certain initiatives. 

#### Evaluation

##### ROUGE

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is primarily designed for and used to evaluate automatic text summarisation systems. It acts as a metric to evaluate the quality of a machine-generated summary by comparing it to one or more human-generated reference summaries.

It calculates the similarity by looking at the overlap of N-grams (consecutive words) or common subsequences between the system-generated summary and the human reference.

It emphasises how much of the reference summary content is captured

In [14]:
import evaluate

# ROUGE
rouge = evaluate.load("rouge")

rouge_scores = rouge.compute(
    predictions=predictions,
    references=references
)

rouge_scores

{'rouge1': np.float64(0.2172932697854816),
 'rouge2': np.float64(0.09123892117931931),
 'rougeL': np.float64(0.16094736001278057),
 'rougeLsum': np.float64(0.19167931410922065)}

Note that here, we have not calculated the ROUGE score for the actual summarisation task, instead we are simply matching it with reference answers for the other 3 tasks. As expected, it only matches the N-gram sequences and the score might not be that great, even if the sentences are accurate semantically.

Calculation of the ROUGE score for the summarisation task is left for the learner as an exercise. You have already seen how it is calculated above. (You might have to write a summary of the transcript on your own as well!)

##### BLEU (precision oriented, strict)

BLEU (Bilingual Evaluation Understudy) is primarily used for machine translation, focusing on precision to measure how much of the generated text matches the reference translation. While ROUGE prioritises recall (capturing content), BLEU emphasises exact n-gram overlap, fluency, and word order, often penalising short outputs via a brevity penalty.

It is highly sensitive to word order, fluency, and exact phrasing, and works well for evaluating machine translation (e.g., Google Translate) and sometimes image captioning or dialogue systems.

BLEU can be applied to summaries, but it is not well aligned with summarisation goals, especially for abstractive summaries where valid paraphrases may have low n-gram overlap. It is more suited to translation tasks

In [ ]:
# BLEU (expects list of reference lists)
bleu = evaluate.load("bleu")

bleu_scores = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

bleu_scores

{'bleu': 0.03884600292129613,
 'precisions': [0.12540894220283533,
  0.05689277899343545,
  0.026344676180021953,
  0.012114537444933921],
 'brevity_penalty': 1.0,
 'length_ratio': 5.878205128205129,
 'translation_length': 917,
 'reference_length': 156}

You can see multiple precisions as it matches on multiple levels of granularity, using different n-gram sequence sizes (1, 2, 3, or 4.) The returned results are aggregates across tasks.

Again, the tasks do not exactly match the objective of BLEU score. You saw how we can calculate the BLUE score, now try evaluating a translation from the LLM using it. (Compare the original sentence with the translated one.)

##### BERTScore (semantic similarity)

BERTScore is a semantic evaluation metric for text generation tasks such as summarisation, machine translation, and text generation. 

Unlike ROUGE or BLEU, BERTScore does not rely on exact n-gram overlap. Instead, it measures semantic similarity using contextual embeddings from a pretrained transformer model (e.g. BERT, RoBERTa).

In [ ]:
# BERTScore
bertscore = evaluate.load("bertscore")

bertscore_scores = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en"
)

bertscore_scores

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'precision': [0.8117866516113281, 0.8355432748794556, 0.8046296834945679],
 'recall': [0.8855541348457336, 0.8931580781936646, 0.9061341881752014],
 'f1': [0.8470674157142639, 0.8633905649185181, 0.8523706793785095],
 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=5.2.0)'}

Here, each list entry corresponds to one candidate-reference pair (for each task) in our evaluation batch, unlike BLEU.

**Interpretation:**

<br>

**ROUGE and BLEU**

The ROUGE scores are **moderate** (ROUGE-1 ≈ 0.22, ROUGE-L ≈ 0.16), indicating partial lexical overlap with the reference. This suggests that the model captures some key content but diverges in phrasing and structure. ROUGE-2 is lower, which is expected when the generated text does not closely match reference bigrams.

The BLEU score is **very low** (≈ 0.04), primarily because the generated output is **much longer** than the reference (length ratio ≈ 5.9). BLEU heavily penalises such cases, even when the content is relevant, because higher-order n-gram precision drops sharply for verbose or paraphrased outputs. This reflects a mismatch in brevity and phrasing rather than a complete failure of task understanding.

Obviously, we have not calculated these metrics on the actual summarisation task, which will lead to much different results.

<br>

**BERTScore**

BERTScore F1 values are **high** (≈ 0.85), with recall consistently higher than precision. This indicates that:

* Most of the semantic content in the reference is present in the generated text (high recall).
* Additional or loosely related information is also included (lower precision).

This pattern is typical of semantically aligned outputs and explains the disagreement with ROUGE and BLEU, which is expected in case of tasks like reasoning or instruction-following.